## Bert 模型的介绍

In [1]:
# 导入包
from transformers import BertForSequenceClassification, AutoTokenizer, BertForTokenClassification, BertForMaskedLM

D:\envs\anaconda3\envs\edu_qa_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = BertForSequenceClassification.from_pretrained("../rag_qa/core/bert_query_classifier")

In [9]:
print("模型架构概览：")
model
#(bert BERT 的核心底座，已经经过海量数据训练过): BertModel（包含 Embeddings 和 12层 Encoder）
#(classifier 分类头): Linear(in_features=768, out_features=2) 需要微调训练的

模型架构概览：


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(21128, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [10]:
# 加载模型和分词器
model_path = "../rag_qa/core/bert_query_classifier"
# model = BertForSequenceClassification.from_pretrained(model_path)  # 加载模型
# 加载分词器 (Tokenizer)
tokenizer = AutoTokenizer.from_pretrained(model_path)


In [5]:
# 输出bert 模型的架构
# BERT->pooler->classifier
model.bert

# (embeddings): 单词、位置、句子类型的向量层
# (encoder): 12层 BertLayer（Self-Attention 和 FeedForward 都在这里）
# (pooler): 一个用于提取 [CLS] 标记的全连接层

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(21128, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [6]:
# 单个句子编码
text = "黑马的Java课程安排有哪些？"
inputs = tokenizer(text, return_tensors="pt") # 返回pytorch
print("原始文本：", text)
print("分词结果：", tokenizer.tokenize(text))
print("Token ID映射：", inputs['input_ids'])

# [101] 是 [CLS]（分类标记），[102] 是 [SEP]（句子结束）

# 【新增】逆向映射
print("ID转回Token：", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0].tolist()))

原始文本： 黑马的Java课程安排有哪些？
分词结果： ['黑', '马', '的', '[UNK]', '课', '程', '安', '排', '有', '哪', '些', '？']
Token ID映射： tensor([[ 101, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,  763,
         8043,  102]])
ID转回Token： ['[CLS]', '黑', '马', '的', '[UNK]', '课', '程', '安', '排', '有', '哪', '些', '？', '[SEP]']


In [7]:
# 句子长度不同，向量长度不同的时候； 填充补全（对齐）
texts = ["黑马的Java课程安排有哪些？", "黑马的Python有哪些？"]
tokenizer(texts, return_tensors="pt", padding=True)


{'input_ids': tensor([[ 101, 7946, 7716, 4638,  100, 6440, 4923, 2128, 2961, 3300, 1525,  763,
         8043,  102],
        [ 101, 7946, 7716, 4638,  100, 3300, 1525,  763, 8043,  102,    0,    0,
            0,    0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0]])}

In [7]:
# 测试模型输出
text = "黑马的Java课程怎么样" # 意图分类 专业知识
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs) # **表示将字典中的键值对作为参数传递给函数
outputs
# Logits 是“未归一化的对数概率”
# 输出结果中，大的就是预测结果；[-6.6753,  6.8557] 还没有做归一化，所以相当于预测为1

SequenceClassifierOutput(loss=None, logits=tensor([[-6.6753,  6.8557]], grad_fn=<AddmmBackward0>), hidden_states=None, attentions=None)

In [8]:
# 获取预测结果
outputs.logits.argmax(dim=-1)

tensor([1])